# ML Predictors for Moran Process — RESIDUAL targets

Twin of `ml_predictors.ipynb`. Everything is identical except the **target**: instead of
predicting a graph's raw fixation probability and time, it predicts the *residual* against
the complete graph of the same size and r — the amplifier/suppressor signal itself.

| raw notebook target | this notebook's target | shape |
|---|---|---|
| `prob_fixation` | `delta_prob_fixation` = rho − rho_complete | signed difference |
| `mean_steps` (logged) | `log_ratio_mean_steps` = log(T / T_complete) | already a log-ratio |

The complete-graph baselines come from `analysis_utils.theory` (exact tridiagonal solve).
They are **not** stored in `graph_statistics.csv` — they are recomputed from `n_nodes` and
`r` on load via `add_analytic_reference_columns`, which is the one extra step below.

> **Read the R² next to the raw notebook's, but mind the r-slice.** At a single r and with
> n almost constant (mostly n=31 here), `fc_*` is nearly a constant, so the residual is
> basically the raw target shifted by a constant — and shifting a target does not change
> R². The residual reframing earns its keep across **varying n and r** (`R_FILTER = None`),
> where the baseline actually moves. Kept at `1.1` here for a clean one-to-one comparison.

**To run:** set `BATCH_NAME` in Setup, run top to bottom. Models save to
`{BATCH_DIR}/ml_models_residual/`.


In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import sys

sys.path.insert(0, "src")
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
# from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import joblib

from moran_process.analysis.analysis_utils import (
    GRAPH_PROPERTY_COLUMNS,
    add_analytic_reference_columns,
)
from moran_process.analysis.analysis_utils.plots import (
    _stamp_batch,
)  # same source-stamp used by every project figure
from moran_process.analysis.analysis_utils.theory import *

# ── Configuration -- only edit these ──────────────────────────────────────────
BATCH_NAME = (
    # "2026_06_21-respiratory-vs-random-redo"  # must match experiment_analysis.ipynb
    "2026_07_15-respiratory-vs-random-100K-reps-2"
)
TARGET_COLUMNS = ["delta_prob_fixation", "log_ratio_mean_steps"]  # residual targets
R_FILTER = 1.1  # None = train across all r values (r becomes a feature)
# e.g. 1.1 = train on one r slice only
DROP_COLLINEAR = True  # prune near-duplicate features so LR coefs / importances
# are interpretable (18 -> 9 features). Set False for full set.
LOG_TARGETS = []  # nothing to log here. log_ratio_mean_steps is ALREADY
# log(T_graph / T_complete), and delta_prob_fixation is a signed difference
# (can be negative), so neither target may be re-logged.
# ──────────────────────────────────────────────────────────────────────────────

# Removed due to collinearity. Each is a near-duplicate (|r|>0.9, VIF in the 1000s)
# of a kept representative:
#   avg_degree, avg_degree_centrality, n_edges  ~ density
#   max_degree_centrality           ~ max_degree
#   diameter, radius, avg_betweenness_centrality, avg/max_closeness_centrality
#                                   ~ average_shortest_path_length
#   transitivity                    ~ average_clustering
# (Dropping min_degree too would push every VIF < 10; kept for distinct leaf-node info.)
COLUMNS_TO_REMOVE = [
    "avg_degree",
    "avg_degree_centrality",
    "n_edges",
    "max_degree_centrality",
    "diameter",
    "radius",
    "avg_betweenness_centrality",
    "avg_closeness_centrality",
    "max_closeness_centrality",
    "transitivity",
]

ROOT = Path(os.getcwd())
BATCH_DIR = ROOT / "simulation_data" / BATCH_NAME
DATA_PATH = BATCH_DIR / "graph_statistics.csv"
# Residual models get their own subdir so this notebook's saved .joblib files and
# Cross-Model Summary never mix with the raw ml_predictors.ipynb ones.
MODEL_SUBDIR = "ml_models_residual"


def stamp_source():
    """Stamp 'source: <batch>' on the current figure (bottom-right), for reproducibility.

    Call right before plt.show(). For SHAP plots, pass show=False to the SHAP call first
    so the stamp lands on the populated figure rather than an empty one.
    """
    _stamp_batch(plt.gcf(), BATCH_NAME)


def target_label(target):
    """Axis label that flags log-space targets."""
    return f"log({target})" if target in LOG_TARGETS else target


def r_tag(r_filter):
    """Filename token recording which r-slice a model was trained on: 'r1.1' for a
    single-r model, 'rall' when r was a feature (R_FILTER is None). Keeps the single-r
    and all-r residual models from overwriting each other in ml_models_residual/."""
    return "rall" if r_filter is None else f"r{r_filter}"


print(f"Batch   : {BATCH_NAME}")
print(f"Targets : {TARGET_COLUMNS}  (log: {LOG_TARGETS})")
print(f"r       : {'all' if R_FILTER is None else R_FILTER}")
print(f"features: {'collinear-pruned' if DROP_COLLINEAR else 'full'}")

In [ ]:
# Drop graph-type-specific metadata and columns too sparse for general prediction
drop_raw = [
    "graph6_string",
    "branching",
    "depth",
    "n_rods",
    "rods_length",
    "rod_length",
    "seed",
    "n_grouped",
]

df = pd.read_csv(DATA_PATH).drop(columns=drop_raw, errors="ignore")
# graph_statistics.csv does not persist the fc_* baselines or the residual targets
# (they are cheap to recompute from n_nodes + r). Add them here, the one step this
# notebook has that the raw one does not.
df = add_analytic_reference_columns(df)
print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
if R_FILTER is not None:
    df_model = df[df["r"] == R_FILTER].copy()
    feature_cols = [c for c in GRAPH_PROPERTY_COLUMNS if c in df.columns]
else:
    df_model = df.copy()
    feature_cols = ["r"] + [c for c in GRAPH_PROPERTY_COLUMNS if c in df.columns]

# ── Random-only modelling ─────────────────────────────────────────────────────
# The model is trained AND scored on the Random graphs only. The handful of structured
# graphs (Avian, Fish, Mammalian, Complete, Star, Cycle, Grid, Line) are held out
# entirely: never seen in training, and shown in the Cross-Model Summary as a labelled
# overlay rather than folded into the metric.
#   Why not put them in the test set? They are ~11 graphs vs 12,000 random ones. Pooled,
#   they move the R²/RMSE by <1% (drowned out); isolated, R² over 11 heterogeneous points
#   is unstable (tiny, arbitrary Var(y) denominator -> can go negative even when RMSE is
#   fine). Their scientific value is per-point (where does each land vs the model?), which
#   a scatter shows and an aggregate hides. So: metric on random, biology as an overlay.
df_random = df_model[df_model["category"] == "Random"].copy()
df_bio = df_model[df_model["category"] != "Random"].copy()

X = df_random[feature_cols].select_dtypes(include=[np.number])
if DROP_COLLINEAR:
    X = X.drop(columns=[c for c in COLUMNS_TO_REMOVE if c in X.columns])
X = X.fillna(X.median())

# Split the RANDOM graphs by topology (wl_hash). In the single-r setup this is one row
# per topology, so it is an ordinary random split; the grouping still matters for the
# all-r variant (keeps every r-row of a graph on one side -> no topology leakage).
from sklearn.model_selection import GroupShuffleSplit

groups = df_random.loc[X.index, "wl_hash"]
train_idx, test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, groups=groups)
)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
leaked = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])

bio_cats = ", ".join(f"{c}×{n}" for c, n in df_bio["category"].value_counts().items())
print(
    f"Random      : {X.shape[0]} graphs x {X.shape[1]} features "
    f"({'collinear-pruned' if DROP_COLLINEAR else 'full'})"
)
print(f"Train / Test: {len(X_train)} / {len(X_test)}  ({len(leaked)} shared topologies, want 0)")
print(f"Held-out bio: {len(df_bio)} graphs  [{bio_cats}]")
print(f"NaNs        : {X.isna().sum().sum()} total")
print("Features    :", list(X.columns))

In [ ]:
# ── Variance-deflation diagnostic ─────────────────────────────────────────────
# Explains why the residual R² looks WORSE than the raw ml_predictors.ipynb even
# though the models are just as accurate. R² = 1 - MSE/Var(y): subtracting the
# complete-graph baseline barely touches MSE (so RMSE is ~unchanged) but shrinks
# Var(y) sharply, so the SAME accuracy reports as a smaller R². The LR/XGBoost
# summary cells below print the raw target's spread next to the residual's so the
# shrinking denominator is explicit rather than mysterious.
RAW_COUNTERPART = {
    "delta_prob_fixation": "prob_fixation",
    "log_ratio_mean_steps": "mean_steps",  # residual is a log-ratio -> compare log(raw)
}


def spread_context(target):
    """'std(residual) vs std(raw counterpart)' on the SAME test rows, as one line."""
    res = df_model.loc[X_test.index, target]
    raw = df_model.loc[X_test.index, RAW_COUNTERPART[target]]
    if target == "log_ratio_mean_steps":
        raw = np.log(raw)  # residual lives in log space; compare like-for-like
    return (
        f"std(target) {res.std():.3g}  vs  raw {raw.std():.3g}  "
        f"-> residual spread is {res.std() / raw.std():.0%} of raw "
        f"(shrinks the R² denominator, not RMSE)"
    )

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ── Presentation helper: horizontal feature-contribution bar chart ────────
# One chart per model, largest-magnitude feature on top. signed=True colours bars by
# sign (red = pushes the target down, blue = up), used for LR standardized coefficients.
# Importance measures (gain, mean|SHAP|) are non-negative, so they use a single colour.
def PRETTY(name):
    return name.replace("_", " ").title()


def plot_feature_bars(values, title, xlabel, signed=False, top_n=5):
    s = values.reindex(values.abs().sort_values(ascending=False).index).head(top_n)
    s = s.iloc[::-1]  # barh draws bottom-up; reverse so the largest sits on top
    fig, ax = plt.subplots(figsize=(8.5, 0.65 * len(s) + 1.4))
    colors = ["#c0392b" if v < 0 else "#2874a6" for v in s] if signed else "#2874a6"
    ax.barh([PRETTY(i) for i in s.index], s.values, color=colors)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_title(title, fontsize=17)
    ax.tick_params(axis="both", labelsize=14)
    ax.grid(axis="x", linestyle="--", alpha=0.35)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    stamp_source()
    plt.show()


def square_lims(true, pred, pred_pct=(1, 99)):
    """(lo, hi) for a squared predicted-vs-actual panel. Always shows the FULL true
    range (so no real data, including the amplifier tail, is cut), but clips the
    predictions to their [1, 99] percentiles so a few over/under-predictions don't
    stretch the axis into whitespace. Use the same lo/hi on x and y with equal aspect
    so the y=x line is a genuine 45°."""
    true = np.asarray(true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    lo = min(true.min(), np.percentile(pred, pred_pct[0]))
    hi = max(true.max(), np.percentile(pred, pred_pct[1]))
    pad = 0.03 * (hi - lo)
    return lo - pad, hi + pad


# ── Real-unit reconstruction of the residual predictions (for interpretable axes) ──
#   delta_prob_fixation -> prob_fixation   = delta + fc_prob_fixation
#   log_ratio_mean_steps -> mean_steps     = fc_fixation_time * exp(log_ratio)
# Metrics note: RMSE here is the honest real-unit accuracy. R² is FOOTNOTED only --
# adding the analytic fc_* baseline back injects size-driven variance the model never
# had to learn, which inflates R² (same effect as the residual-vs-raw discussion). The
# residual panels keep the honest R².
REAL_UNIT = {
    "delta_prob_fixation": {
        "real": "prob_fixation",
        "base": "fc_prob_fixation",
        "invert": lambda p, b: p + b,
        "log": False,
        "unit": "fixation probability ρ",
    },
    "log_ratio_mean_steps": {
        "real": "mean_steps",
        "base": "fc_fixation_time",
        "invert": lambda p, b: b * np.exp(p),
        "log": True,  # spans ~27x, so log-log axes
        "unit": "mean fixation time (steps)",
    },
}


def plot_real_units(target, preds, model_name, cmap):
    cfg = REAL_UNIT[target]
    base = df_model.loc[X_test.index, cfg["base"]].to_numpy()
    true = df_model.loc[X_test.index, cfg["real"]].to_numpy()
    pred = cfg["invert"](np.asarray(preds, dtype=float), base)
    # Metrics are the RESIDUAL-space ones (what the model actually predicts) -- the
    # same honest numbers as the residual panel. Recomputing R²/RMSE on the real values
    # would report the baseline-inflated figures we agreed NOT to trust. For probability
    # the residual RMSE equals the real-ρ RMSE (fc cancels); for steps the residual RMSE
    # is in log units, so it is labelled as such rather than shown as a steps count.
    resid_true = df_model.loc[X_test.index, target].to_numpy()
    resid_pred = np.asarray(preds, dtype=float)
    r2 = r2_score(resid_true, resid_pred)
    rmse = root_mean_squared_error(resid_true, resid_pred)
    metric_note = "residual-space" + (" (log)" if cfg["log"] else "")

    if cfg["log"]:
        lo = max(min(true.min(), np.percentile(pred, 1)), 1e-9) * 0.9
        hi = max(true.max(), np.percentile(pred, 99)) * 1.1
        scale, extent = "log", None
    else:
        lo, hi = square_lims(true, pred)
        scale, extent = "linear", (lo, hi, lo, hi)

    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(true, pred, gridsize=30, cmap=cmap, mincnt=1,
                   xscale=scale, yscale=scale, extent=extent)
    ax.plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect prediction")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    cax = make_axes_locatable(ax).append_axes("right", size="4.5%", pad=0.12)
    fig.colorbar(hb, cax=cax, label="Count")
    ax.set_xlabel(f"Actual {cfg['unit']}")
    ax.set_ylabel(f"Predicted {cfg['unit']}")
    ax.set_title(f"{model_name}: Predicted vs Actual — {cfg['unit']}")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3, which="both")
    ax.text(0.05, 0.95,
            f"{metric_note}\n$R^2$ = {r2:.3f}\nRMSE = {rmse:.3g}",
            transform=ax.transAxes, fontsize=10, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    stamp_source()
    plt.show()



# ── Presentation styling for SHAP beeswarms ───────────────────────────────
# Larger fonts for slides, and a top-k trim so a beeswarm shows exactly k feature rows
# (no "Sum of N other features" row, which is noise on a presentation).
SHAP_TOPK = 5
SHAP_FONT = {
    "font.size": 15, "axes.titlesize": 17, "axes.labelsize": 16,
    "xtick.labelsize": 14, "ytick.labelsize": 15,
}


def top_k_explanation(expl, k=SHAP_TOPK):
    """Keep only the k features with the largest mean|SHAP| so a beeswarm of the result
    shows exactly k rows and no aggregate 'other features' row."""
    order = np.argsort(np.abs(expl.values).mean(axis=0))[::-1][:k]
    return expl[:, list(order)]


## Linear Regression Baseline

Standardizes features then fits a linear model. Standardized coefficients give a direct interpretability signal -- the largest absolute values are the graph properties that most linearly drive the target. Compare these with the XGBoost SHAP values below: agreement between the two is stronger evidence that a property genuinely matters.

In [ ]:
lr_models = {}
lr_artifacts = {}

for target in TARGET_COLUMNS:
    y_train = df_model.loc[X_train.index, target]
    y_test = df_model.loc[X_test.index, target]
    if target in LOG_TARGETS:  # model in log space; preds stay in log space
        y_train, y_test = np.log(y_train), np.log(y_test)

    std_lr = make_pipeline(StandardScaler(), LinearRegression())
    std_lr.fit(X_train, y_train)
    lr_preds = std_lr.predict(X_test)

    coef_df = pd.DataFrame(
        {
            "feature": X.columns,
            "std_coefficient": std_lr.named_steps["linearregression"].coef_,
        }
    )

    model_filename = (
        BATCH_DIR / MODEL_SUBDIR / f"{target}_{r_tag(R_FILTER)}_linear_regression_pipeline.joblib"
    )
    os.makedirs(os.path.dirname(model_filename), exist_ok=True)
    joblib.dump(std_lr, model_filename)

    lr_models[target] = std_lr
    lr_artifacts[target] = {
        "preds": lr_preds,
        "y_test": y_test,
        "y_train": y_train,
        "coef_df": coef_df,
    }
    print(
        f"[{target}] Saved LR model ({r_tag(R_FILTER)})."
        f"{'  (log-space)' if target in LOG_TARGETS else ''}"
    )

In [ ]:
for target, art in lr_artifacts.items():
    space = " (log-space)" if target in LOG_TARGETS else ""
    print(f"\n── {target}{space} ──")
    print(f"  R²  : {r2_score(art['y_test'], art['preds']):.4f}")
    print(f"  RMSE: {root_mean_squared_error(art['y_test'], art['preds']):.4f}")
    print(f"  {spread_context(target)}")

    coef_df = art["coef_df"].copy()
    coef_df["abs_coefficient"] = coef_df["std_coefficient"].abs()
    coef_df["contribution_pct"] = (
        coef_df["abs_coefficient"] / coef_df["abs_coefficient"].sum() * 100
    )
    top = coef_df.sort_values("contribution_pct", ascending=False).copy()
    top["feature"] = top["feature"].str.replace("_", " ").str.title()
    print(
        top[["feature", "contribution_pct", "std_coefficient"]]
        .head(10)
        .to_string(float_format=lambda x: f"{x:.2f}")
    )

In [ ]:
# ── LR: which variables drive each prediction ────────────────────────
# Features are standardized, so |coefficient| is the per-feature contribution and the
# sign is the direction of effect. Same numbers as the printed table above, as a slide.
for target, art in lr_artifacts.items():
    coefs = art["coef_df"].set_index("feature")["std_coefficient"]
    plot_feature_bars(
        coefs,
        title=f"LR feature contribution — {PRETTY(target)}",
        xlabel="Standardized coefficient (signed)",
        signed=True,
    )

In [ ]:
for target, art in lr_artifacts.items():
    lbl = target_label(target)
    y_test, preds = art["y_test"], art["preds"]
    lo, hi = square_lims(y_test, preds)

    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(y_test, preds, gridsize=30, cmap="Blues", mincnt=1, extent=(lo, hi, lo, hi))
    ax.plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect prediction")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    # Colorbar height tracks the SQUARE axes: make_axes_locatable recomputes its box at
    # draw time, so it follows the aspect-driven shrink instead of overhanging it.
    cax = make_axes_locatable(ax).append_axes("right", size="4.5%", pad=0.12)
    fig.colorbar(hb, cax=cax, label="Count")
    ax.set_xlabel(f"Actual {lbl}")
    ax.set_ylabel(f"Predicted {lbl}")
    ax.set_title(f"LR: Predicted vs Actual ({lbl})")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    r2 = r2_score(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    ax.text(
        0.05, 0.95, f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.3g}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )
    stamp_source()
    plt.show()

In [ ]:
# Same LR predictions as the panel above, reconstructed into real units (ρ, steps).
# See plot_real_units: honest metric is RMSE (real units); R² is footnoted only.
for target, art in lr_artifacts.items():
    plot_real_units(target, art["preds"], "LR", cmap="Blues")

In [ ]:
# Compute LR SHAP values (LinearExplainer is exact for a linear model:
# phi_i = beta_i * (x_i - mean)). Rendered as a beeswarm in the next cell; kept here as
# compute-only so the same beeswarm is not drawn twice (summary_plot defaulted to it).
lr_shap = {}

for target, art in lr_artifacts.items():
    scaler = lr_models[target].named_steps["standardscaler"]
    lr_model = lr_models[target].named_steps["linearregression"]

    X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

    explainer = shap.LinearExplainer(lr_model, X_train_scaled)
    lr_shap[target] = explainer(X_test_scaled)


In [ ]:
# LR SHAP beeswarm: top 5 features, presentation fonts.
for target, shap_vals in lr_shap.items():
    with plt.rc_context(SHAP_FONT):
        shap.plots.beeswarm(top_k_explanation(shap_vals), max_display=SHAP_TOPK, show=False)
        plt.title(f"LR SHAP beeswarm ({target})")
        stamp_source()
        plt.show()


## XGBoost Model

A gradient-boosted tree ensemble. Unlike LR, it captures non-linear interactions between graph properties without needing feature scaling. If XGBoost dramatically outperforms LR, that suggests the relationship between graph structure and fixation outcome is non-linear.

In [ ]:
xgb_models = {}
xgb_artifacts = {}

for target in TARGET_COLUMNS:
    y_train = df_model.loc[X_train.index, target]
    y_test = df_model.loc[X_test.index, target]
    if target in LOG_TARGETS:  # model in log space; preds stay in log space
        y_train, y_test = np.log(y_train), np.log(y_test)

    xgb_model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        objective="reg:squarederror",
        n_jobs=1,
        random_state=42,
        base_score=y_train.mean(),
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)

    model_filename = (
        BATCH_DIR / MODEL_SUBDIR / f"{target}_{r_tag(R_FILTER)}_xgboost_model.joblib"
    )
    joblib.dump(xgb_model, model_filename)

    xgb_models[target] = xgb_model
    xgb_artifacts[target] = {"preds": xgb_preds, "y_test": y_test, "y_train": y_train}
    print(
        f"[{target}] Saved XGBoost model ({r_tag(R_FILTER)})."
        f"{'  (log-space)' if target in LOG_TARGETS else ''}"
    )

In [ ]:
for target, art in xgb_artifacts.items():
    lbl = target_label(target)
    space = " (log-space)" if target in LOG_TARGETS else ""
    y_test, preds = art["y_test"], art["preds"]
    r2 = r2_score(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    print(f"\n── {target}{space} ──")
    print(f"  R²  : {r2:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  {spread_context(target)}")

    lo, hi = square_lims(y_test, preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(y_test, preds, gridsize=30, cmap="Reds", mincnt=1, extent=(lo, hi, lo, hi))
    ax.plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect prediction")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    cax = make_axes_locatable(ax).append_axes("right", size="4.5%", pad=0.12)
    fig.colorbar(hb, cax=cax, label="Count")
    ax.set_xlabel(f"Actual {lbl}")
    ax.set_ylabel(f"Predicted {lbl}")
    ax.set_title(f"XGBoost: Predicted vs Actual ({lbl})")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    ax.text(
        0.05, 0.95, f"$R^2$ = {r2:.3f}\nRMSE = {rmse:.3g}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )
    stamp_source()
    plt.show()

In [ ]:
# Same XGBoost predictions as the panel above, reconstructed into real units.
for target, art in xgb_artifacts.items():
    plot_real_units(target, art["preds"], "XGBoost", cmap="Reds")

In [ ]:
# ── XGBoost: native GAIN importance ──────────────────────────────────────
# GAIN = total split-quality improvement each feature gave the trees. Model-internal,
# non-negative, and NOT on the SHAP scale: it answers "how useful was this feature for
# splitting", a different question from the beeswarm's "how much did it move predictions".
# (The mean|SHAP| bar that used to sit here was dropped -- it is just the beeswarm's
# per-point SHAP collapsed to its average, so the beeswarm below already conveys it.)
for target in TARGET_COLUMNS:
    model = xgb_models[target]

    gain = (
        pd.Series(model.get_booster().get_score(importance_type="gain"), dtype=float)
        .reindex(X.columns)
        .fillna(0.0)  # features never split on contribute 0 gain
    )
    plot_feature_bars(
        gain,
        title=f"XGBoost gain importance ({PRETTY(target)})",
        xlabel="Gain (split-quality improvement)",
    )


In [ ]:
# ── XGBoost SHAP beeswarm via XGBoost's own exact tree-SHAP ───────────────
# shap.TreeExplainer(model) CANNOT be used here: SHAP 0.49 + XGBoost 3.x store
# base_score as a JSON array string ('[-0.142]'), which SHAP's parser feeds to
# float() and crashes on at construction time (verified on this env). XGBoost
# computes the identical TreeSHAP in C++ via pred_contribs=True; we wrap the result
# in a shap.Explanation so every shap.plots.* view works -- exact, and in seconds
# (no interventional background -> no O(N^2) blow-up that caused the ~15 min run).
for target in TARGET_COLUMNS:
    booster  = xgb_models[target].get_booster()
    contribs = booster.predict(xgb.DMatrix(X_test), pred_contribs=True)
    values, base = contribs[:, :-1], contribs[:, -1]  # last column = base/bias term

    expl = shap.Explanation(
        values=values,
        base_values=base,
        data=X_test.to_numpy(),
        feature_names=list(X_test.columns),
    )
    with plt.rc_context(SHAP_FONT):
        shap.plots.beeswarm(top_k_explanation(expl), max_display=SHAP_TOPK, show=False)
        plt.title(f"XGBoost SHAP beeswarm ({target})")
        stamp_source()
        plt.show()

    top = X_test.columns[np.abs(values).mean(0).argmax()]
    print(f"[{target}] Top feature: {top}")


## Model Agreement

If LR and XGBoost make similar predictions despite very different inductive biases, the signal is likely robust rather than an artifact of one model's assumptions.

In [ ]:
for target in TARGET_COLUMNS:
    lbl = target_label(target)
    lr_preds = lr_artifacts[target]["preds"]
    xgb_preds = xgb_artifacts[target]["preds"]

    # Both axes are predictions: base the square on both clouds (1-99 pct) + equal aspect
    # so the "full agreement" y=x line sits at a true 45°.
    lo = min(np.percentile(lr_preds, 1), np.percentile(xgb_preds, 1))
    hi = max(np.percentile(lr_preds, 99), np.percentile(xgb_preds, 99))
    pad = 0.03 * (hi - lo)
    lo, hi = lo - pad, hi + pad

    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(lr_preds, xgb_preds, gridsize=30, cmap="Purples", mincnt=1, extent=(lo, hi, lo, hi))
    ax.plot([lo, hi], [lo, hi], "r--", lw=2, label="Full agreement")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    cax = make_axes_locatable(ax).append_axes("right", size="4.5%", pad=0.12)
    fig.colorbar(hb, cax=cax, label="Count")
    ax.set_xlabel(f"LR prediction ({lbl})")
    ax.set_ylabel(f"XGBoost prediction ({lbl})")
    ax.set_title(f"LR vs XGBoost: {lbl}")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    corr, p_value = pearsonr(lr_preds, xgb_preds)
    ax.text(
        0.05, 0.95, f"Pearson r = {corr:.4f}\np = {p_value:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )
    stamp_source()
    plt.show()

In [ ]:
# ── Model agreement in REAL units (ρ, steps) ──────────────────────────────────
# Same LR-vs-XGBoost comparison as the residual-space panel above, but both predictions
# are pushed back through the analytic complete-graph baseline of each test row:
#   delta_prob_fixation  -> rho   = delta + fc_prob_fixation
#   log_ratio_mean_steps -> steps = fc_fixation_time * exp(log_ratio)
# Mirrors the plot_real_units() pairing used for the predicted-vs-actual panels.
# Metrics: the headline Pearson r is the RESIDUAL-space one (agreement on what the models
# actually predict). The real-unit r is footnoted because adding the shared fc_* baseline
# back injects a common, size-driven component into BOTH axes and inflates the
# correlation -- the same denominator effect discussed for R² elsewhere in this notebook.
def plot_agreement_real_units(target, lr_preds, xgb_preds, cmap="Purples"):
    cfg = REAL_UNIT[target]
    base = df_model.loc[X_test.index, cfg["base"]].to_numpy()
    lr_real = cfg["invert"](np.asarray(lr_preds, dtype=float), base)
    xgb_real = cfg["invert"](np.asarray(xgb_preds, dtype=float), base)

    if cfg["log"]:  # spans orders of magnitude -> log-log, percentile-clipped square
        lo = max(min(np.percentile(lr_real, 1), np.percentile(xgb_real, 1)), 1e-9) * 0.9
        hi = max(np.percentile(lr_real, 99), np.percentile(xgb_real, 99)) * 1.1
        scale, extent = "log", None
    else:
        lo = min(np.percentile(lr_real, 1), np.percentile(xgb_real, 1))
        hi = max(np.percentile(lr_real, 99), np.percentile(xgb_real, 99))
        pad = 0.03 * (hi - lo)
        lo, hi = lo - pad, hi + pad
        scale, extent = "linear", (lo, hi, lo, hi)

    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(lr_real, xgb_real, gridsize=30, cmap=cmap, mincnt=1,
                   xscale=scale, yscale=scale, extent=extent)
    ax.plot([lo, hi], [lo, hi], "r--", lw=2, label="Full agreement")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    cax = make_axes_locatable(ax).append_axes("right", size="4.5%", pad=0.12)
    fig.colorbar(hb, cax=cax, label="Count")
    ax.set_xlabel(f"LR prediction ({cfg['unit']})")
    ax.set_ylabel(f"XGBoost prediction ({cfg['unit']})")
    ax.set_title(f"LR vs XGBoost — {cfg['unit']}")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3, which="both")

    corr, p_value = pearsonr(lr_preds, xgb_preds)              # residual space (honest)
    corr_real, _ = pearsonr(lr_real, xgb_real)                 # real units (baseline-inflated)
    ax.text(
        0.05, 0.95,
        f"residual-space\nPearson r = {corr:.4f}\np = {p_value:.2e}\n"
        f"(real-unit r = {corr_real:.4f})",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )
    stamp_source()
    plt.show()


for target in TARGET_COLUMNS:
    plot_agreement_real_units(target, lr_artifacts[target]["preds"], xgb_artifacts[target]["preds"])


## Cross-Model Summary

Loads all saved models and evaluates them on the full dataset. Run this section only after training both targets (`prob_fixation` and `mean_steps`) -- all four `.joblib` files must exist in `{BATCH_DIR}/ml_models/`.

In [ ]:
from sklearn.metrics import r2_score
import matplotlib.lines as mlines
import seaborn as sns
from moran_process.analysis.analysis_utils import (
    CATEGORY_COLOR_DICT,
    generate_robust_color_dict,
)

ML_MODELS_DIR = BATCH_DIR / MODEL_SUBDIR

MODEL_TYPE_MAP = {
    "linear_regression": "LR",
    "xgboost": "XGBoost",
}

# Only load models trained on the CURRENT r-slice (r_tag), so the single-r and all-r
# residual models can coexist in the dir without the loader mixing them up.
tag = r_tag(R_FILTER)
all_models = {}
for path in sorted(ML_MODELS_DIR.glob(f"*_{tag}_*.joblib")):
    stem = path.stem  # e.g. 'delta_prob_fixation_r1.1_xgboost_model'
    model_type_key = next((k for k in MODEL_TYPE_MAP if k in stem), None)
    if model_type_key is None:
        print(f"Skipping unrecognised model file: {path.name}")
        continue
    target = stem[: stem.index(f"_{tag}_")]  # everything before the r tag
    model_type = MODEL_TYPE_MAP[model_type_key]
    all_models[(target, model_type)] = joblib.load(path)

expected_features = list(next(iter(all_models.values())).feature_names_in_)
print(f"Loaded {len(all_models)} models:")
for target, model_type in sorted(all_models):
    print(f"  {target:20} | {model_type}")

In [ ]:
# ── Figure 1 of 2: model quality on the held-out RANDOM fold ───────────────────
# Pure LR-vs-XGBoost comparison, predicted vs actual, residual-space metric. No organs
# here (they are Figure 2), so both models sit on equal, tight axes and the LR column is
# not distorted by its extrapolation on the structured graphs. Blues = LR, Reds = XGBoost
# (same colormaps as the per-model panels).
df_test = df_random.loc[X_test.index].copy()
HEXBIN_CMAP = {"LR": "Blues", "XGBoost": "Reds"}
unique_targets = sorted(set(t for t, _ in all_models))
unique_model_types = sorted(set(m for _, m in all_models))


def hexbin_panel(ax, true, pred, r2, rmse, metric_note, unit, cmap, log):
    true = np.asarray(true, float)
    pred = np.asarray(pred, float)
    if log:
        lo = max(min(true.min(), np.percentile(pred, 1)), 1e-9) * 0.9
        hi = max(true.max(), np.percentile(pred, 99)) * 1.1
        scale, extent = "log", None
    else:
        lo, hi = square_lims(true, pred)
        scale, extent = "linear", (lo, hi, lo, hi)
    ax.hexbin(true, pred, gridsize=35, cmap=cmap, mincnt=1,
              xscale=scale, yscale=scale, extent=extent)
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5)
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    ax.set_xlabel(f"True {unit}", fontsize=11)
    ax.set_ylabel(f"Predicted {unit}", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.3, which="both")
    ax.text(0.05, 0.95, f"{metric_note}\n$R^2$ = {r2:.3f}\nRMSE = {rmse:.3g}",
            transform=ax.transAxes, fontsize=11, verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9))


slice_label = "all r" if R_FILTER is None else f"r = {R_FILTER}"
fig, axes = plt.subplots(len(unique_targets), len(unique_model_types),
                         figsize=(8 * len(unique_model_types), 6 * len(unique_targets)),
                         squeeze=False)
fig.suptitle(f"Model quality on held-out random graphs ({slice_label})", fontsize=16)
for ri, target in enumerate(unique_targets):
    cfg = REAL_UNIT[target]
    base = df_test[cfg["base"]].to_numpy()
    true = df_test[cfg["real"]].to_numpy()
    resid_true = df_test[target].to_numpy()
    for ci, mt in enumerate(unique_model_types):
        ax = axes[ri, ci]
        if (target, mt) not in all_models:
            ax.set_visible(False)
            continue
        resid_pred = np.asarray(all_models[(target, mt)].predict(X_test), float)
        pred = cfg["invert"](resid_pred, base)
        r2 = r2_score(resid_true, resid_pred)
        rmse = root_mean_squared_error(resid_true, resid_pred)
        hexbin_panel(ax, true, pred, r2, rmse,
                     "residual-space" + (" (log)" if cfg["log"] else ""),
                     cfg["unit"], HEXBIN_CMAP.get(mt, "Greys"), cfg["log"])
        ax.set_title(f"{mt}: {cfg['unit']}", fontsize=13)
plt.subplots_adjust(left=0.07, right=0.97, top=0.92, bottom=0.07, wspace=0.2, hspace=0.3)
_stamp_batch(fig, BATCH_NAME)
plt.show()

In [ ]:
# ── Figure 2 of 2: where the biological graphs land (XGBoost) ──────────────────
# XGBoost only -- it does not extrapolate, so the structured graphs stay on sane axes
# (LR would fling Complete/Star to impossible values). Grey hexbin = held-out random fold
# (the reference population); each colored dot = one biological/structured graph.
# Reading the panel:
#   * HORIZONTAL (true value vs the grey cloud): right of the cloud = amplifier (higher ρ
#     / longer time than random graphs of its size), left = suppressor.
#   * VERTICAL distance from the y=x line: model error. On the line = a random-trained
#     model predicts this graph from topology alone; off the line = the model misses it
#     (structure the features do not capture, or extrapolation beyond the random range).
# Complete is omitted: the residual is defined AGAINST the complete graph, so its residual
# is ~0 by construction, and it is 4 near-identical points.
BIO_MODEL = "XGBoost"
OVERLAY_EXCLUDE = {"Complete"}
df_test = df_random.loc[X_test.index].copy()
df_overlay = df_bio[~df_bio["category"].isin(OVERLAY_EXCLUDE)].copy()
train_median = X.loc[X_train.index].median()
X_overlay = df_overlay[list(X.columns)].fillna(train_median)
category_color_dict = generate_robust_color_dict(df_model, CATEGORY_COLOR_DICT)

targets_here = [t for t in unique_targets if (t, BIO_MODEL) in all_models]
slice_label = "all r" if R_FILTER is None else f"r = {R_FILTER}"
fig, axes = plt.subplots(1, len(targets_here), figsize=(8 * len(targets_here), 6.5), squeeze=False)
fig.suptitle(f"Biological graphs vs random baseline — {BIO_MODEL} ({slice_label})", fontsize=16)
for ci, target in enumerate(targets_here):
    cfg = REAL_UNIT[target]
    ax = axes[0, ci]
    log = cfg["log"]
    model = all_models[(target, BIO_MODEL)]
    rand_base = df_test[cfg["base"]].to_numpy()
    rand_true = df_test[cfg["real"]].to_numpy()
    rand_pred = cfg["invert"](np.asarray(model.predict(X_test), float), rand_base)
    bio_base = df_overlay[cfg["base"]].to_numpy()
    bio_true = df_overlay[cfg["real"]].to_numpy()
    bio_pred = cfg["invert"](np.asarray(model.predict(X_overlay), float), bio_base)
    # axis spans the random true values and every organ point
    lo = min(rand_true.min(), np.percentile(rand_pred, 1), bio_true.min(), bio_pred.min())
    hi = max(rand_true.max(), np.percentile(rand_pred, 99), bio_true.max(), bio_pred.max())
    if log:
        lo, hi = max(lo, 1e-9) * 0.9, hi * 1.1
    else:
        pad = 0.03 * (hi - lo)
        lo, hi = lo - pad, hi + pad
    scale = "log" if log else "linear"
    extent = None if log else (lo, hi, lo, hi)
    ax.hexbin(rand_true, rand_pred, gridsize=35, cmap="Greys", mincnt=1,
              xscale=scale, yscale=scale, extent=extent)
    for xt, yp, cat in zip(bio_true, bio_pred, df_overlay["category"]):
        ax.scatter(xt, yp, s=300, marker="o", c=category_color_dict.get(cat, "gray"),
                   edgecolor="k", linewidth=1.5, zorder=5)
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5)
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    ax.set_title(cfg["unit"], fontsize=13)
    ax.set_xlabel(f"True {cfg['unit']}", fontsize=11)
    ax.set_ylabel(f"Predicted {cfg['unit']}", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.3, which="both")

present_cats = sorted(df_overlay["category"].dropna().unique())
handles = [
    mlines.Line2D([], [], marker="o", color="w", markerfacecolor=category_color_dict.get(c, "gray"),
                  markeredgecolor="k", markersize=11, label=c)
    for c in present_cats
]
handles.append(mlines.Line2D([], [], marker="h", color="w", markerfacecolor="0.5",
                             markersize=12, label="Random (density)"))
handles.append(mlines.Line2D([], [], color="k", linestyle="--", alpha=0.5, label="Ideal (y=x)"))
fig.legend(handles=handles, title="Graph type", loc="center left",
           bbox_to_anchor=(0.90, 0.5), fontsize=11, title_fontsize=12)
plt.subplots_adjust(left=0.07, right=0.88, top=0.90, bottom=0.10, wspace=0.25)
_stamp_batch(fig, BATCH_NAME)
plt.show()

In [ ]:
# ── Figure 2b: biology overlay WITH the held-out random-fold metric box ────────
# Same as Figure 2, but each panel carries the residual-space R²/RMSE on the held-out
# random fold (the honest model-quality number). The point of the box: it certifies the
# model IS accurate (~0.83 on ρ, ~0.94 on time) so the organs sitting OFF the diagonal
# read as signal (out-of-distribution amplifier/suppressor behaviour), not model error.
BIO_MODEL = "XGBoost"
OVERLAY_EXCLUDE = {"Complete"}
df_test = df_random.loc[X_test.index].copy()
df_overlay = df_bio[~df_bio["category"].isin(OVERLAY_EXCLUDE)].copy()
train_median = X.loc[X_train.index].median()
X_overlay = df_overlay[list(X.columns)].fillna(train_median)
category_color_dict = generate_robust_color_dict(df_model, CATEGORY_COLOR_DICT)

targets_here = [t for t in unique_targets if (t, BIO_MODEL) in all_models]
slice_label = "all r" if R_FILTER is None else f"r = {R_FILTER}"
fig, axes = plt.subplots(1, len(targets_here), figsize=(8 * len(targets_here), 6.5), squeeze=False)
fig.suptitle(f"Biological graphs vs random baseline — {BIO_MODEL} ({slice_label})", fontsize=16)
for ci, target in enumerate(targets_here):
    cfg = REAL_UNIT[target]
    ax = axes[0, ci]
    log = cfg["log"]
    model = all_models[(target, BIO_MODEL)]
    rand_base = df_test[cfg["base"]].to_numpy()
    rand_true = df_test[cfg["real"]].to_numpy()
    resid_true = df_test[target].to_numpy()
    resid_pred = np.asarray(model.predict(X_test), float)          # residual space
    rand_pred = cfg["invert"](resid_pred, rand_base)               # -> real units
    r2 = r2_score(resid_true, resid_pred)                          # held-out random fold
    rmse = root_mean_squared_error(resid_true, resid_pred)
    metric_note = "residual-space" + (" (log)" if log else "")
    bio_base = df_overlay[cfg["base"]].to_numpy()
    bio_true = df_overlay[cfg["real"]].to_numpy()
    bio_pred = cfg["invert"](np.asarray(model.predict(X_overlay), float), bio_base)
    lo = min(rand_true.min(), np.percentile(rand_pred, 1), bio_true.min(), bio_pred.min())
    hi = max(rand_true.max(), np.percentile(rand_pred, 99), bio_true.max(), bio_pred.max())
    if log:
        lo, hi = max(lo, 1e-9) * 0.9, hi * 1.1
    else:
        pad = 0.03 * (hi - lo)
        lo, hi = lo - pad, hi + pad
    scale = "log" if log else "linear"
    extent = None if log else (lo, hi, lo, hi)
    ax.hexbin(rand_true, rand_pred, gridsize=35, cmap="Greys", mincnt=1,
              xscale=scale, yscale=scale, extent=extent)
    for xt, yp, cat in zip(bio_true, bio_pred, df_overlay["category"]):
        ax.scatter(xt, yp, s=300, marker="o", c=category_color_dict.get(cat, "gray"),
                   edgecolor="k", linewidth=1.5, zorder=5)
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5)
    if log:
        ax.set_xscale("log")
        ax.set_yscale("log")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", "box")
    ax.set_title(cfg["unit"], fontsize=13)
    ax.set_xlabel(f"True {cfg['unit']}", fontsize=11)
    ax.set_ylabel(f"Predicted {cfg['unit']}", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.3, which="both")
    ax.text(0.05, 0.95, f"{metric_note}, random fold\n$R^2$ = {r2:.3f}\nRMSE = {rmse:.3g}",
            transform=ax.transAxes, fontsize=11, verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9))

present_cats = sorted(df_overlay["category"].dropna().unique())
handles = [
    mlines.Line2D([], [], marker="o", color="w", markerfacecolor=category_color_dict.get(c, "gray"),
                  markeredgecolor="k", markersize=11, label=c)
    for c in present_cats
]
handles.append(mlines.Line2D([], [], marker="h", color="w", markerfacecolor="0.5",
                             markersize=12, label="Random (density)"))
handles.append(mlines.Line2D([], [], color="k", linestyle="--", alpha=0.5, label="Ideal (y=x)"))
fig.legend(handles=handles, title="Graph type", loc="center left",
           bbox_to_anchor=(0.90, 0.5), fontsize=11, title_fontsize=12)
plt.subplots_adjust(left=0.07, right=0.88, top=0.90, bottom=0.10, wspace=0.25)
_stamp_batch(fig, BATCH_NAME)
plt.show()

# Now Let's test it on other batches


In [ ]:
# Score the models trained above (on BATCH_NAME, loaded as `all_models`) on a DIFFERENT
# batch. Only the evaluation data changes -- the models stay. Set TEST_BATCH and run.
#   e.g. BATCH_NAME = scaling batch (random graphs)  +  TEST_BATCH = respiratory batch
#        -> "does a model trained only on random graphs predict the lung graphs?"
# Requires the Cross-Model Summary cells above to have run (defines all_models,
# expected_features, unique_targets, unique_model_types).
TEST_BATCH = "2026_06_18-respiratory-vs-random"  # batch whose graphs we score
HIGHLIGHT = {
    "Avian",
    "Fish",
    "Mammalian",
    "Complete",
    "Cycle",
    "Star",
    "Grid",
}  # outline non-random types

test_df = pd.read_csv(ROOT / "simulation_data" / TEST_BATCH / "graph_statistics.csv")
test_df = add_analytic_reference_columns(test_df)  # same recompute as the training batch
test_df = test_df if R_FILTER is None else test_df[test_df["r"] == R_FILTER].copy()
X_test_batch = test_df[expected_features].fillna(test_df[expected_features].median())

# Coverage: are the test graphs inside the TRAINING feature ranges? (X is the training
# feature matrix from the setup cell.) Out-of-range features mean the model extrapolates.
print(f"Coverage of '{TEST_BATCH}' vs '{BATCH_NAME}' training ranges:")
for f in expected_features:
    out = int(((X_test_batch[f] < X[f].min()) | (X_test_batch[f] > X[f].max())).sum())
    print(
        f"  {f:30} train[{X[f].min():.3g}, {X[f].max():.3g}]"
        + ("" if out == 0 else f"   <-- {out} graphs extrapolating")
    )

test_colors = generate_robust_color_dict(test_df, CATEGORY_COLOR_DICT)

fig, axes = plt.subplots(
    len(unique_targets),
    len(unique_model_types),
    figsize=(8 * len(unique_model_types), 6 * len(unique_targets)),
    squeeze=False,
)
fig.suptitle(f"Transfer: {BATCH_NAME} models  ->  {TEST_BATCH}", fontsize=15)

for r_i, target in enumerate(unique_targets):
    log_t = target in LOG_TARGETS
    y_true = np.log(test_df[target]) if log_t else test_df[target].astype(float)
    for c_i, model_type in enumerate(unique_model_types):
        ax = axes[r_i, c_i]
        if (target, model_type) not in all_models:
            ax.set_visible(False)
            continue
        preds = all_models[(target, model_type)].predict(X_test_batch)
        for cat in sorted(test_df["category"].dropna().unique()):
            mask = (test_df["category"] == cat).values
            big = cat in HIGHLIGHT
            ax.scatter(
                y_true[mask],
                preds[mask],
                s=180 if big else 25,
                c=test_colors.get(cat, "gray"),
                edgecolor="k" if big else "none",
                linewidth=1.2 if big else 0,
                alpha=0.9 if big else 0.4,
                zorder=3 if big else 1,
            )
        lims = [min(y_true.min(), preds.min()), max(y_true.max(), preds.max())]
        ax.plot(lims, lims, "k--", alpha=0.6)
        lbl = target_label(target)
        # Report BOTH transfer metrics. On the thin n=30 prob_fixation band R² goes
        # negative (target variance ~ model error) even though RMSE is tiny, so RMSE is
        # the honest accuracy number here; R² just measures "beats predicting the mean".
        r2 = r2_score(y_true, preds)
        rmse = root_mean_squared_error(y_true, preds)
        ax.set_title(
            f"{model_type}: {lbl}\n(transfer $R^2$={r2:.3f}, RMSE={rmse:.3g})",
            fontsize=12,
        )
        ax.set_xlabel(f"True {lbl}")
        ax.set_ylabel(f"Predicted {lbl}")
        ax.grid(alpha=0.3)

handles = [
    mlines.Line2D(
        [],
        [],
        marker="o",
        color="w",
        markerfacecolor=test_colors.get(c, "gray"),
        markeredgecolor="k",
        markersize=10,
        label=c,
    )
    for c in sorted(test_df["category"].dropna().unique())
]
handles.append(
    mlines.Line2D([], [], color="k", linestyle="--", alpha=0.6, label="Ideal (y=x)")
)
fig.legend(
    handles=handles, title="Category", loc="center left", bbox_to_anchor=(0.91, 0.5)
)
_stamp_batch(fig, f"train:{BATCH_NAME}  test:{TEST_BATCH}")
plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()